# NB12 — EVALUATION3 manual calibration 500 + optional 500

Mục tiêu của notebook này:

1. Reuse candidate pools + human evidence đã có.
2. Generate **500 threshold-calibration pairs** mới, không trùng các pair đã review.
3. Có **một cell optional** để generate thêm 500 pair thứ hai nếu batch đầu chưa đủ.
4. Có **retrieval-audit cell riêng** cho các candidate nằm ngoài gate `pHash <= 4` (`6/8/10`), không trộn lẫn với threshold calibration.
5. `SAME_PRODUCT_DIFFERENT_IMAGE` được map thành `DUPLICATE` theo definition hiện tại của nhóm.

Không chạy full 28k E3 và không thay đổi protocol chính. Đây là công cụ tạo human evidence/calibration.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

from google.colab import drive, auth
drive.mount('/content/drive', force_remount=False)
auth.authenticate_user()

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
BRANCH = 'feat/evaluation3-manual-calibration-v2'
REPO_ROOT = Path('/content/opisoverated-e3-manual-v2')

def run_git(*args, cwd=None):
    return subprocess.run(['git','-c','http.version=HTTP/1.1',*args], cwd=cwd, check=True, text=True)

if not (REPO_ROOT/'.git').is_dir():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_ROOT))
else:
    run_git('fetch','origin',BRANCH,cwd=REPO_ROOT)
    run_git('switch',BRANCH,cwd=REPO_ROOT)
    run_git('pull','--ff-only','origin',BRANCH,cwd=REPO_ROOT)

subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'datasets','pandas','pillow','scikit-image','openpyxl','gspread'
], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.evaluation3_generate_manual_calibration_v2 import (
    generate_batch, build_human_evidence, evidence_retrieval_report,
    collect_new_human_labels, threshold_diagnostics, labeled_retrieval_report,
)
print('BRANCH:', BRANCH)


In [ ]:
# Paths + local E3 extraction (avoid slow/brittle Drive FUSE image reads)
DRIVE = Path('/content/drive/MyDrive')
E3_DIR = DRIVE/'EVALUATION3'
THRESHOLD_ROOT = E3_DIR/'phash_ssim_threshold'
OUTPUT_ROOT = E3_DIR/'manual_calibration_v2'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP = E3_DIR/'outfit.zip'
LOCAL_ZIP = Path('/content/evaluation3_outfit.zip')
LOCAL_EXTRACT = Path('/content/evaluation3_manual_v2')

if not LOCAL_ZIP.is_file():
    print('Copying outfit.zip to local Colab disk...')
    shutil.copy2(ZIP, LOCAL_ZIP)

if not LOCAL_EXTRACT.exists():
    print('Extracting locally...')
    shutil.unpack_archive(str(LOCAL_ZIP), str(LOCAL_EXTRACT), format='zip')

candidates = [LOCAL_EXTRACT/'Outfits', LOCAL_EXTRACT/'outfit', LOCAL_EXTRACT]
E3_ROOT = next((p for p in candidates if p.is_dir() and any(p.iterdir())), None)
if E3_ROOT is None:
    raise FileNotFoundError('Could not locate extracted E3 image root')

# Load the 300 already-reviewed hard-negative labels from Google Sheet.
import google.auth, gspread, pandas as pd
creds, _ = google.auth.default()
gc = gspread.authorize(creds)
HARD_REVIEW_SHEET_ID = '1b9pj-E_C0BMfyN8s6upKxzcfntR_T77QAfLMwBDBu_g'
ws = gc.open_by_key(HARD_REVIEW_SHEET_ID).worksheet('hard_negative_review_BLIND')
hard_labels = pd.DataFrame(ws.get_all_records())

# Preserve the 14 current borderline pairs that the team manually confirmed as DUPLICATE.
EXTRA_EVIDENCE = []
borderline_key = E3_DIR/'eval3_test_overlap'/'borderline_review_KEY.csv'
if borderline_key.is_file():
    bk = pd.read_csv(borderline_key)
    confirmed14 = bk[
        bk['audit_status'].astype(str).eq('BORDERLINE')
        & bk['matched_polyvore_item_id'].notna()
    ].copy()
    if len(confirmed14):
        confirmed14['eval3_rel_path'] = (
            'Outfits/' + confirmed14['outfit_id'].astype(str)
            + '/' + confirmed14['role'].astype(str) + '.jpg'
        )
        confirmed14['polyvore_item_id'] = confirmed14['matched_polyvore_item_id'].astype(str)
        confirmed14['human_label'] = 'DUPLICATE'
        confirmed14['phash_distance'] = confirmed14['best_phash_distance']
        confirmed14['ssim'] = confirmed14['best_ssim']
        confirmed_path = OUTPUT_ROOT/'manual_borderline_14_confirmed.csv'
        confirmed14[
            ['eval3_rel_path','polyvore_item_id','human_label','phash_distance','ssim']
        ].to_csv(confirmed_path, index=False)
        EXTRA_EVIDENCE.append(confirmed_path)
        print('Preserved manually-confirmed borderline DUP pairs:', len(confirmed14))

evidence = build_human_evidence(
    THRESHOLD_ROOT,
    hard_review_labels=hard_labels,
    extra_evidence_csvs=EXTRA_EVIDENCE,
)
print('E3_ROOT:', E3_ROOT)
print('Existing human evidence:', len(evidence))
print(evidence.human_label.value_counts().to_dict())
print(evidence_retrieval_report(evidence))


## Batch 1 — 500 pair cho threshold calibration

Batch này chỉ lấy `pHash <= 4`, oversample vùng SSIM boundary và có high/low controls.
Các pair human-evidence cũ (400 positive + 300 hard review + 14 borderline confirmed) tự động bị exclude.
SSIM trong KEY được **recompute bằng RGB 256px thumbnail + white-pad**, cùng preprocessing với pipeline hiện tại.


In [ ]:
report_1 = generate_batch(
    threshold_root=THRESHOLD_ROOT,
    e3_root=E3_ROOT,
    output_root=OUTPUT_ROOT,
    batch_number=1,
    batch_size=500,
    hard_review_labels=hard_labels,
    extra_evidence_csvs=EXTRA_EVIDENCE,
    mode='threshold',
)
report_1


## OPTIONAL — Batch 2 thêm 500 pair

**Không chạy cell này ngay.** Chỉ đổi `RUN_SECOND_500=True` nếu sau khi label batch 1,
distribution vẫn chưa đủ để chốt threshold. Batch 2 tự loại toàn bộ pair của batch 1 và human evidence cũ.


In [ ]:
RUN_SECOND_500 = False

if RUN_SECOND_500:
    report_2 = generate_batch(
        threshold_root=THRESHOLD_ROOT,
        e3_root=E3_ROOT,
        output_root=OUTPUT_ROOT,
        batch_number=2,
        batch_size=500,
        hard_review_labels=hard_labels,
        extra_evidence_csvs=EXTRA_EVIDENCE,
        mode='threshold',
    )
    print(report_2)
else:
    print('SKIPPED batch 2. Chỉ bật khi batch 1 chưa đủ.')


## Retrieval audit — làm trước khi freeze SSIM threshold

Đây là queue **riêng**, không dùng để tune SSIM threshold trực tiếp.
Nó lấy candidate đang bị gate `pHash <= 4` chặn: mặc định `pHash = 6/8/10`,
ưu tiên pair có similarity hint cao.

Lưu ý: pHash median-balanced hiện tại thường tạo Hamming distance chẵn, nên “audit 5–8”
thực tế gần như là `6` và `8`. Human evidence cũ còn có confirmed DUP ở `pHash=10`,
nên notebook cho audit tới 10.


In [ ]:
RUN_RETRIEVAL_AUDIT = False

if RUN_RETRIEVAL_AUDIT:
    retrieval_report = generate_batch(
        threshold_root=THRESHOLD_ROOT,
        e3_root=E3_ROOT,
        output_root=OUTPUT_ROOT,
        batch_number=1,
        batch_size=500,
        hard_review_labels=hard_labels,
        extra_evidence_csvs=EXTRA_EVIDENCE,
        mode='retrieval',
        retrieval_max_phash=10,
    )
    print(retrieval_report)
else:
    print('SKIPPED retrieval audit. Bật khi nhóm sẵn sàng review pHash 6/8/10.')


## Sau khi label xong

Mở `*_BLIND.xlsx`, điền `DUPLICATE` / `NON_DUPLICATE`, save lại.
Cell dưới chỉ tạo bảng diagnostic; do sampling được stratify quanh boundary,
không được coi class prevalence trong batch là prevalence thật của population.


In [ ]:
RUN_DIAGNOSTICS = False

if RUN_DIAGNOSTICS:
    labeled = collect_new_human_labels(OUTPUT_ROOT, prefix='threshold_batch_')
    print('New labeled threshold pairs:', len(labeled))
    if len(labeled):
        display(
            threshold_diagnostics(labeled, phash_max=4)
            .sort_values(['duplicate_precision','duplicate_recall_on_review_sample'], ascending=False)
            .head(30)
        )

    retrieval = labeled_retrieval_report(OUTPUT_ROOT)
    if len(retrieval):
        display(retrieval)
else:
    print('SKIPPED diagnostics.')
